In [ ]:
!pip install langgraph langchain python-dotenv typing langchain_google_genai

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict

In [ ]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [ ]:
class Blog_State(TypedDict):

  title: str
  outline: str
  content: str

In [ ]:
def outline(state: Blog_State) -> Blog_State:

  title = state["title"]
  prompt = f'generate outline for blog on topic {title}'
  outline = model.invoke(prompt)
  state["outline"] = outline

  return state

def blog(state: Blog_State) -> Blog_State:

  title = state["title"]
  outline = state["outline"]
  prompt = f'write blog on topic {title} with outline {outline}'
  content = model.invoke(prompt)
  state['content'] = content

  return state

In [ ]:
graph = StateGraph(Blog_State)

graph.add_node('create_outline', outline)
graph.add_node('create_blog', blog)

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

In [ ]:
in_state = {"title": "LLM transformers"}

output = workflow.invoke(in_state)

print(output['content'])